# Mint your own work with Misty DOI

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zistgah/hecon/blob/main/notebooks/06_mint_with_misty.ipynb)

How this paper was published, and how you can give your own teaching cases, notebooks or papers a DOI with a provenance trail you control.

**How to use it.** Press ▶ on each step in order, or choose Runtime, then Run all. Each step's code is hidden behind its title; double-click the title to see it. Educational use only: nothing here is a medical device or a finding.

© 1993–2026 Abhishek Choudhary. All rights reserved. AyeAI. Code GPL-3.0-or-later; text CC BY-SA 4.0. AyeAI® and AyeAM® are registered trade marks of Abhishek Choudhary; AyeCNSe™ (Indian TM application No. 7415871, pending). Cite with [CITATION.cff](https://github.com/zistgah/hecon/blob/main/CITATION.cff); the DOI is recorded in the repository README once minted.

**Where this notebook lives.** It is hosted in zistgah/hecon until it moves to its home, project-ilm/misty-doi, the estate's DOI and MASI tooling. The move and the link back are listed in [COMPONENTS.md](https://github.com/zistgah/hecon/blob/main/COMPONENTS.md).

## The process

The estate follows one order for anything it publishes: **seal** (SHA-256 manifest), **clear** (no secrets or personal data), **attest** (the author's typed intent, recorded and timestamped with OpenTimestamps) and **mint** (a DOI through Zenodo with Misty DOI). MASI, Misty's scholarly process, runs from concept and narrative through metadata, repository seeding and DOI minting to SHA-256 and OpenTimestamps proofs and long-term preservation. A *concept* DOI names the work across versions; a *version* DOI names one release. A DOI is a dated public disclosure: if a venue matters to you, check its preprint policy first (an optional clause in the estate's contract; the author of this paper minted for provenance).

## Step 1: install Misty DOI

From PyPI, as the estate's own tools do. Nothing is installed outside this runtime.

In [ ]:
#@title Install misty-doi { display-mode: "form" }
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "misty-doi"], check=True)
print(subprocess.run(["misty", "--version"], capture_output=True, text=True).stdout.strip())

## Step 2: describe your work

One canonical record, `misty.json`, drives the DOI and every citation format. Names are written *Family, Given*.

In [ ]:
#@title Write misty.json { display-mode: "form" }
import json
title = "A case-based learning module on hyperkalaemia in haemodialysis"  #@param {type:"string"}
creator = "Family, Given"  #@param {type:"string"}
affiliation = "Your institution"  #@param {type:"string"}
orcid = ""  #@param {type:"string"}
upload_type = "lesson"  #@param ["lesson", "publication", "software", "dataset", "presentation", "poster"]
licence = "cc-by-sa-4.0"  #@param ["cc-by-sa-4.0", "cc-by-4.0", "gpl-3.0", "cc0-1.0"]
description = "A synthetic teaching case with questions mapped to clinical reasoning domains."  #@param {type:"string"}
person = {"name": creator, "affiliation": affiliation}
if orcid: person["orcid"] = orcid
meta = {"title": title, "version": "1.0.0", "upload_type": upload_type, "description": description, "license": licence,
        "access_right": "open", "language": "eng", "creators": [person], "keywords": ["medical education", "clinical reasoning"]}
if upload_type == "publication": meta["publication_type"] = "preprint"
json.dump(meta, open("misty.json", "w"), indent=1)
print(subprocess.run(["misty", "validate", "-m", "misty.json"], capture_output=True, text=True))

## Step 3: every citation format from one record

Misty derives the Zenodo record, DataCite, codemeta and `CITATION.cff`.

In [ ]:
#@title Transform { display-mode: "form" }
r = subprocess.run(["misty", "transform", "-m", "misty.json", "-o", "build"], capture_output=True, text=True)
print(r.stdout[-600:] or r.stderr[-600:])
import os; print(sorted(os.listdir("build")) if os.path.isdir("build") else "no build folder")

## Step 4: package without the network

A dry run builds the deposit package and checksums locally and contacts nobody.

In [ ]:
#@title Dry run { display-mode: "form" }
open("module.md", "w").write("# Module\n\nYour teaching material goes here.\n")
r = subprocess.run(["misty", "publish", "-m", "misty.json", "-f", "module.md", "--dry-run", "--output", "result.json"], capture_output=True, text=True)
print(r.stdout[-400:] or r.stderr[-400:]); print(open("result.json").read()[:400] if os.path.exists("result.json") else "")

## Step 5: follow a journal track

MISTY's MASI ledger records a manuscript's states and refuses steps that skip evidence, such as an acceptance with no recorded review round.

In [ ]:
#@title MASI journal track { display-mode: "form" }
for args in (["new", "my-module", "--track", "journal", "--title", title], ["state", "my-module", "--to", "submitted"], ["status", "my-module"]):
    r = subprocess.run(["misty", "masi"] + args, capture_output=True, text=True); print((r.stdout + r.stderr).strip()[-500:])

## Step 6: timestamp and mint, under your own control

These steps reach the internet and act in your name, so they are off until you switch them on. Minting asks for your Zenodo token (typed, never stored) and for the word MINT. They were not run when this notebook was built.

In [ ]:
#@title Timestamp and mint { display-mode: "form" }
stamp = False  #@param {type:"boolean"}
mint = False  #@param {type:"boolean"}
sandbox = True  #@param {type:"boolean"}
if stamp:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opentimestamps-client"], check=True)
    print(subprocess.run(["misty", "ots", "stamp", "module.md"], capture_output=True, text=True))
if mint:
    from getpass import getpass
    import os
    os.environ["ZENODO_TOKEN"] = getpass("Zenodo token: ")
    if input("Type MINT to publish a DOI: ").strip() == "MINT":
        cmd = ["misty", "publish", "-m", "misty.json", "-f", "module.md", "--output", "minted.json"] + (["--sandbox"] if sandbox else [])
        print(subprocess.run(cmd, capture_output=True, text=True))
    else:
        print("Not minted.")

## How this paper itself was minted

The author runs one script on his own machine, `seed_hecon.sh --all`. It installs the repository through the estate's operations library (zops, in zistgah/governance), asks him to type PUSH, seals and pushes, switches on Pages, asks him to type MINT, mints through `misty publish`, records the DOI back into README.md and CITATION.cff, and seals again. Every typed word is kept as a signed, timestamped receipt in the repository.